# Small-Scale AutoSchemaKG v1 — local Qwen3.5-2B

This notebook runs the first end-to-end version with a local Qwen model. Select a GPU runtime, then run all cells in order. No commercial LLM API key is required.

In [1]:
import os, platform, subprocess, sys
print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
REPO_URL = 'https://github.com/phuongth05/SmallScaledAutoSchemaKG.git'
REPO_DIR = '/content/SmallScaledAutoSchemaKG'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Repository:', os.getcwd())

Repository: /content/SmallScaledAutoSchemaKG


## Install
Qwen3.5 support follows recent vLLM releases, so this cell uses the vLLM nightly wheel index. Installation can take several minutes.

In [3]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([
    'uv', 'pip', 'install', '--system', 'vllm', '--torch-backend=auto',
    '--extra-index-url', 'https://wheels.vllm.ai/nightly'
], check=True)

CompletedProcess(args=['uv', 'pip', 'install', '--system', 'vllm', '--torch-backend=auto', '--extra-index-url', 'https://wheels.vllm.ai/nightly'], returncode=0)

In [6]:
!pip uninstall -y torchaudio torchvision

import subprocess
import sys

subprocess.run([
    sys.executable,
    "-c",
    "import torch, vllm; print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'vLLM', vllm.__version__)"
], check=True)

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.28.0+cu132
Uninstalling torchvision-0.28.0+cu132:
  Successfully uninstalled torchvision-0.28.0+cu132


CompletedProcess(args=['/usr/bin/python3', '-c', "import torch, vllm; print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'vLLM', vllm.__version__)"], returncode=0)

In [9]:
!pip uninstall -y torchaudio torchvision
!uv pip install --system torchvision --torch-backend=cu132

Using Python 3.12.13 environment at: /usr
Resolved 32 packages in 3.02s
Prepared 1 package in 1ms
Installed 1 package in 4ms
 + torchvision==0.28.0+cu132


In [10]:
import torch
import torchvision
import vllm

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("TorchVision:", torchvision.__version__)
print("vLLM:", vllm.__version__)

Torch: 2.13.0+cu132
CUDA: 13.2
TorchVision: 0.28.0+cu132
vLLM: 0.27.2rc1.dev163+g017e9f444


## Start the local model server
The 2B checkpoint is the closest smaller official Qwen3.5 model to the requested 3B scale. Change `MODEL_ID` to `Qwen/Qwen3.5-4B` only if the assigned GPU has enough memory.

In [11]:
import shutil, time
import requests

MODEL_ID = 'Qwen/Qwen3.5-2B'
PORT = 8000
LOG_PATH = '/content/qwen35_vllm.log'
vllm_executable = shutil.which('vllm')
if not vllm_executable:
    raise RuntimeError('vLLM executable was not installed')
server_log = open(LOG_PATH, 'w', encoding='utf-8')
server_cmd = [
    vllm_executable, 'serve', MODEL_ID,
    '--host', '127.0.0.1', '--port', str(PORT),
    '--dtype', 'half', '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.85', '--language-model-only'
]
server = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)
deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        server_log.flush()
        raise RuntimeError(f'vLLM stopped early. Read {LOG_PATH}')
    try:
        response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
        if response.ok:
            print('Local model server is ready:', response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(f'vLLM did not become ready. Read {LOG_PATH}')

Local model server is ready: {'object': 'list', 'data': [{'id': 'Qwen/Qwen3.5-2B', 'object': 'model', 'created': 1786988617, 'owned_by': 'vllm', 'root': 'Qwen/Qwen3.5-2B', 'parent': None, 'max_model_len': 8192, 'permission': [{'id': 'modelperm-9e5484b10e7d8e1d', 'object': 'model_permission', 'created': 1786988617, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


## Run extraction, schema induction, and GraphML export

In [12]:
OUTPUT_DIR = '/content/colab_outputs/v1'
subprocess.run([
    sys.executable, 'scripts/run_colab_v1.py',
    '--model', MODEL_ID,
    '--base-url', f'http://127.0.0.1:{PORT}/v1',
    '--output-dir', OUTPUT_DIR,
    '--overwrite'
], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_colab_v1.py', '--model', 'Qwen/Qwen3.5-2B', '--base-url', 'http://127.0.0.1:8000/v1', '--output-dir', '/content/colab_outputs/v1', '--overwrite'], returncode=0)

In [13]:
import json
from pathlib import Path
summary = json.loads(Path(OUTPUT_DIR, 'run_summary.json').read_text())
print(json.dumps(summary, indent=2))
assert summary.get('nodes', 0) > 0, 'The graph contains no nodes'
assert summary.get('edges', 0) > 0, 'The graph contains no edges'

{
  "model": "Qwen/Qwen3.5-2B",
  "data": "/content/SmallScaledAutoSchemaKG/example/example_data/v1_smoke.json",
  "output_directory": "/content/colab_outputs/v1",
  "include_concepts": true,
  "graphml": "/content/colab_outputs/v1/kg_graphml/v1_smoke_graph.graphml",
  "nodes": 217,
  "edges": 257,
  "node_types": {
    "entity": 7,
    "event": 4,
    "passage": 1,
    "concept": 205
  }
}


## Package the outputs
Download the resulting zip from the Colab file browser.

In [14]:
archive = shutil.make_archive('/content/autoschemakg_colab_v1', 'zip', OUTPUT_DIR)
print('Created:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

Created: /content/autoschemakg_colab_v1.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
from google.colab import drive
drive.mount("/content/drive/")

!cp /content/autoschemakg_colab_v1.zip \
    /content/drive/MyDrive/NLP-final/autoschemakg_colab_v1.zip

Mounted at /content/drive/
